# 2.0 · P2 — Definición del riesgo por estación

**Objetivo:** elegir el horizonte H y la fracción f de la etiqueta de riesgo de P2:

> En la hora *t*, una estación está en riesgo si |llegadas − salidas en *t+1 … t+H*| ≥ *f* × capacidad (`dpcapacity`).
> Flujo neto negativo = **sin_bicis**; positivo = **sin_anclajes**.

Todo se calcula **solo con TRAIN** (ene–sep 2025) para no usar información de validación ni de prueba.
Meta: clase positiva entre ~2 % y 10 % (lo bastante rara para ser una alerta y lo bastante frecuente para aprenderla).

Kernel: el del `.venv` del proyecto (`import mibici` debe funcionar sin modificar `sys.path`).

In [1]:
import pandas as pd

from mibici.config import (
    ESTACIONES_PARQUET,
    P2_MAX_DURATION_SECONDS,
    P2_MIN_DURATION_SECONDS,
    P2_TEST_END,
    P2_TRAIN_START,
    P2_VAL_START,
    VIAJES_PARQUET,
)
from mibici.features import p2_agregar_etiqueta, p2_construir_panel

estaciones = pd.read_parquet(ESTACIONES_PARQUET)
capacidad = estaciones.set_index("id")["dpcapacity"]
print(estaciones["dpcapacity"].describe().round(1).to_string())
print("Estaciones en catálogo:", len(estaciones), "| con capacidad 0:", int((capacidad == 0).sum()))

count    484.0
mean      16.9
std        4.1
min        0.0
25%       15.0
50%       15.0
75%       19.0
max       39.0
Estaciones en catálogo: 484 | con capacidad 0: 4


## 1. Panel estación × hora restringido a TRAIN

In [2]:
viajes = pd.read_parquet(
    VIAJES_PARQUET, columns=["inicio_viaje", "fin_viaje", "origen_id", "destino_id"]
)
duracion = (viajes["fin_viaje"] - viajes["inicio_viaje"]).dt.total_seconds()
viajes = viajes.loc[duracion.between(P2_MIN_DURATION_SECONDS, P2_MAX_DURATION_SECONDS)].dropna()

panel = p2_construir_panel(viajes, P2_TRAIN_START, P2_TEST_END)
train = panel.loc[panel.index.get_level_values("hora") < pd.Timestamp(P2_VAL_START)]  # SOLO TRAIN

est_train = set(train.index.get_level_values("estacion"))
print("Estaciones con viajes en TRAIN:", len(est_train))
print("  sin catálogo:", len(est_train - set(capacidad.index)))
print("  con capacidad 0:", len(est_train & set(capacidad[capacidad == 0].index)))

Estaciones con viajes en TRAIN: 369
  sin catálogo: 0
  con capacidad 0: 0


## 2. Prevalencia de la etiqueta por horizonte × fracción (TRAIN)

In [3]:
filas = []
for h in (1, 2, 3):
    for f in (0.3, 0.4, 0.5):
        etiquetado = p2_agregar_etiqueta(train, h, capacidad, f)
        tipo = etiquetado["tipo_riesgo"].value_counts(normalize=True)
        filas.append(
            {
                "horizonte_h": h,
                "fraccion": f,
                "riesgo": etiquetado["riesgo"].mean(),
                "sin_bicis": tipo.get("sin_bicis", 0.0),
                "sin_anclajes": tipo.get("sin_anclajes", 0.0),
            }
        )
prevalencia = pd.DataFrame(filas)
prevalencia.style.format({c: "{:.2%}" for c in ("riesgo", "sin_bicis", "sin_anclajes")})

,horizonte_h,fraccion,riesgo,sin_bicis,sin_anclajes
0,1,0.300000,2.56%,1.31%,1.24%
1,1,0.400000,1.16%,0.60%,0.57%
2,1,0.500000,0.44%,0.22%,0.22%
3,2,0.300000,6.90%,3.54%,3.35%
4,2,0.400000,3.81%,1.97%,1.84%
5,2,0.500000,1.78%,0.91%,0.87%
6,3,0.300000,11.09%,5.70%,5.39%
7,3,0.400000,6.74%,3.50%,3.23%
8,3,0.500000,3.58%,1.87%,1.71%


## 3. ¿Por qué relativo a la capacidad y no un umbral fijo?

Se compara la etiqueta elegida (H = 2, f = 0.4) contra un umbral fijo en número de bicis
(0.4 × capacidad mediana = 6 bicis), por rango de capacidad de la estación.

In [4]:
H, F = 2, 0.4
UMBRAL_FIJO = F * capacidad[capacidad > 0].median()

comp = p2_agregar_etiqueta(train, H, capacidad, F)
comp["riesgo_fijo"] = (comp["neto_futuro"].abs() >= UMBRAL_FIJO).astype("int8")
comp["rango_capacidad"] = pd.cut(
    comp["capacidad"], [0, 12, 15, 19, 40], labels=["≤12", "13–15", "16–19", "≥20"]
)
columnas = {"riesgo": "relativo (f × capacidad)", "riesgo_fijo": f"fijo (≥ {UMBRAL_FIJO:.0f} bicis)"}
por_rango = comp.groupby("rango_capacidad", observed=True)
tabla = por_rango[["riesgo", "riesgo_fijo"]].mean()
tabla["estaciones"] = por_rango.apply(lambda d: d.index.get_level_values("estacion").nunique())
tabla.loc["global"] = [*comp[["riesgo", "riesgo_fijo"]].mean(), tabla["estaciones"].sum()]
formato = {v: "{:.2%}" for v in columnas.values()} | {"estaciones": "{:.0f}"}
tabla.rename(columns=columnas).style.format(formato)

,relativo (f × capacidad),fijo (≥ 6 bicis),estaciones
rango_capacidad,,,
≤12,4.81%,2.57%,47
13–15,4.38%,4.38%,167
16–19,2.69%,5.89%,106
≥20,3.29%,11.12%,49
global,3.81%,5.48%,369


## 4. Decisión

- **H = 2 h, f = 0.4** → clase positiva ≈ 3.8 % en TRAIN, repartida casi por igual entre *sin_bicis* (≈ 2.0 %) y *sin_anclajes* (≈ 1.8 %).
  - H = 2 da margen operativo al equipo de rebalanceo; con H = 1 la clase queda por debajo de 2 % para f ≥ 0.4.
  - f = 0.5 deja la clase en ≈ 1.8 % (bajo la meta); f = 0.3 (≈ 6.9 %) marca movimientos de solo 30 % de la capacidad, que la estación normalmente absorbe.
- **Umbral fijo descartado:** con 6 bicis, las estaciones de ≥ 20 anclajes quedan marcadas 11.1 % del tiempo y las de ≤ 12 solo 2.6 %, aunque a una estación pequeña le bastan menos movimientos para vaciarse o llenarse. Con el umbral relativo la tasa es comparable entre tamaños (2.7 %–4.8 %).
- **Capacidad sola, insuficiente:** es constante por estación; no dice nada de *cuándo* hay riesgo. El riesgo lo produce el flujo neto, y la capacidad solo lo escala.
- **Exclusiones:** el catálogo tiene 4 estaciones con capacidad 0, pero ninguna tiene viajes en TRAIN, y no hay estaciones con viajes sin catálogo: no se pierde ninguna fila.

Estos valores quedan en `mibici/config.py` como `P2_HORIZONTE_HORAS` y `P2_FRACCION_CAPACIDAD`.

## 5. Partición temporal: ¿por qué tres conjuntos en P2?

El equipo acordó una partición temporal ≈ 80/20 sin mezcla aleatoria: **entrenamiento ene–oct** y **prueba nov–dic 2025**.
P2 la respeta, con el **mismo TEST** que P1 y P3, y solo subdivide el bloque de entrenamiento:

- **TRAIN ene–sep:** ajuste de los modelos y del perfil histórico por estación.
- **VALIDACIÓN oct:** elección del umbral de decisión.
- **TEST nov–dic:** evaluación final, una sola vez.

Se descartan las horas cuya ventana de predicción (H = 2 h) cruza la frontera entre conjuntos, y en VALIDACIÓN y TEST solo entran estaciones con viajes en TRAIN.
Las filas se construyen con las mismas funciones que usa `mibici/modeling/p2_clasificacion.py`.

In [5]:
from mibici.config import (
    P2_FRACCION_CAPACIDAD,
    P2_HORIZONTE_HORAS,
    P2_TEST_START,
)
from mibici.features import (
    P2_FEATURES,
    p2_agregar_features,
    p2_ajustar_historico,
    p2_aplicar_historico,
    p2_historico_pasado,
)
from mibici.modeling.p2_clasificacion import cortar

df = p2_agregar_features(
    p2_agregar_etiqueta(panel, P2_HORIZONTE_HORAS, capacidad, P2_FRACCION_CAPACIDAD),
    P2_HORIZONTE_HORAS,
)
conj = {
    "TRAIN": cortar(df, P2_TRAIN_START, P2_VAL_START),
    "VALIDACIÓN": cortar(df, P2_VAL_START, P2_TEST_START),
    "TEST": cortar(df, P2_TEST_START, P2_TEST_END),
}
viajes_train = viajes.loc[viajes["inicio_viaje"] < pd.Timestamp(P2_VAL_START)]
estaciones_train = pd.Index(viajes_train["origen_id"]).union(viajes_train["destino_id"])
for nombre in ("VALIDACIÓN", "TEST"):
    d = conj[nombre]
    conj[nombre] = d.loc[d.index.get_level_values("estacion").isin(estaciones_train)]

historico = p2_ajustar_historico(conj["TRAIN"])  # SOLO TRAIN
conj["TRAIN"] = p2_historico_pasado(conj["TRAIN"])
for nombre in ("VALIDACIÓN", "TEST"):
    conj[nombre] = p2_aplicar_historico(conj[nombre], historico)
conj = {k: d.dropna(subset=P2_FEATURES) for k, d in conj.items()}

total = sum(len(d) for d in conj.values())
periodos = {"TRAIN": "ene–sep 2025", "VALIDACIÓN": "oct 2025", "TEST": "nov–dic 2025"}
particion = pd.DataFrame(
    {
        "periodo": [periodos[k] for k in conj],
        "filas": [len(d) for d in conj.values()],
        "% filas": [len(d) / total for d in conj.values()],
        "estaciones": [d.index.get_level_values("estacion").nunique() for d in conj.values()],
        "prevalencia riesgo": [d["riesgo"].mean() for d in conj.values()],
    },
    index=list(conj),
)
particion.style.format({"filas": "{:,}", "% filas": "{:.1%}", "prevalencia riesgo": "{:.2%}"})

,periodo,filas,% filas,estaciones,prevalencia riesgo
TRAIN,ene–sep 2025,"2,348,476",74.3%,369,3.84%
VALIDACIÓN,oct 2025,"273,798",8.7%,369,3.94%
TEST,nov–dic 2025,"539,478",17.1%,369,2.85%


### El umbral de decisión necesita datos que no sean ni de entrenamiento ni de prueba

P2 es la única pregunta con un **umbral de decisión**: el clasificador da una probabilidad y hay que decidir a partir de cuál se emite la alerta.
Con una clase positiva de ~3–4 % y `class_weight="balanced"`, las probabilidades quedan infladas y el 0.5 por defecto no sirve.

Se entrenan el baseline y el candidato con TRAIN y se compara el desempeño **en TEST** según dónde se elija el umbral (el que maximiza F1).
La fila "TEST" es solo una referencia de lo que daría hacer trampa: elegir el umbral con los datos de prueba es fuga de información.

In [6]:
from sklearn.metrics import f1_score, precision_score, recall_score

from mibici.modeling.p2_clasificacion import (
    construir_baseline,
    construir_candidato,
    elegir_umbral,
)


def matriz(d):
    return d[P2_FEATURES].astype("float64")


y = {k: d["riesgo"] for k, d in conj.items()}
horas_test = conj["TEST"].index.get_level_values("hora").nunique()

filas = []
for nombre_modelo, modelo in (
    ("baseline LogReg", construir_baseline()),
    ("candidato HGB", construir_candidato()),
):
    modelo.fit(matriz(conj["TRAIN"]), y["TRAIN"])
    prob = {k: modelo.predict_proba(matriz(d))[:, 1] for k, d in conj.items()}
    umbrales = {
        "0.5 por defecto": 0.5,
        "TRAIN": elegir_umbral(y["TRAIN"], prob["TRAIN"]),
        "VALIDACIÓN (usado)": elegir_umbral(y["VALIDACIÓN"], prob["VALIDACIÓN"]),
        "TEST (fuga, solo referencia)": elegir_umbral(y["TEST"], prob["TEST"]),
    }
    for origen, umbral in umbrales.items():
        pred = (prob["TEST"] >= umbral).astype(int)
        filas.append(
            {
                "modelo": nombre_modelo,
                "umbral elegido con": origen,
                "umbral": umbral,
                "test_f1": f1_score(y["TEST"], pred),
                "test_recall": recall_score(y["TEST"], pred),
                "test_precision": precision_score(y["TEST"], pred),
                "alertas por hora": pred.sum() / horas_test,
            }
        )

umbrales_tabla = pd.DataFrame(filas).set_index(["modelo", "umbral elegido con"])
umbrales_tabla.style.format(
    {
        "umbral": "{:.3f}",
        "test_f1": "{:.3f}",
        "test_recall": "{:.3f}",
        "test_precision": "{:.3f}",
        "alertas por hora": "{:.1f}",
    }
)

### Conclusión sobre la partición

- **Se respeta el 80/20 del equipo.** El TEST de P2 es el mismo de P1 y P3 (nov–dic, 17 % de las filas); TRAIN + VALIDACIÓN suman el 83 % (ene–oct).
- **Ajustar el umbral es indispensable.** Con 0.5, el candidato tiene recall alto (0.88), pero solo 14 % de sus alertas son correctas: unas 67 alertas por hora, inmanejables para el equipo de rebalanceo. Con el umbral ajustado (≈ 0.88), el F1 en TEST sube de 0.24 a 0.40 y quedan ~14 alertas por hora.
- **El umbral se elige en VALIDACIÓN** porque es la única opción metodológicamente limpia: elegirlo en TEST es fuga y sobrestima el desempeño. En estos datos, elegirlo en TRAIN habría dado casi el mismo resultado (F1 0.402 contra 0.401), porque el perfil histórico de TRAIN se calcula solo con días previos y el modelo no está sobreajustado. Aun así, octubre queda como conjunto limpio para ajustar hiperparámetros en las siguientes entregas sin tocar TEST.
- **El modelo final no se reentrena con TRAIN + VALIDACIÓN:** se evalúa en TEST el mismo modelo entrenado con ene–sep cuyo umbral se eligió en octubre, tal como lo registra MLflow.